In [ ]:
%pip install pandas
import pandas as pd
print(pd.__version__)

In [ ]:
%pip install matplotlib
import matplotlib
import matplotlib.pyplot as plt
print(matplotlib.__version__)

In [ ]:
import zipfile
import pandas as pd
import matplotlib.pyplot as plt

# 1. ระบุตำแหน่งไฟล์ zip
zip_path = r'C:\Users\User\Downloads\archive.zip'

# 2. เปิดและอ่านไฟล์ CSV ที่อยู่ข้างใน zip โดยตรง
with zipfile.ZipFile(zip_path, 'r') as z:
    # ค้นหาชื่อไฟล์ .csv ใน zip
    csv_filename = [f for f in z.namelist() if f.endswith('.csv')][0]
    
    # อ่านไฟล์ข้อมูลด้วย encoding='latin1'
    with z.open(csv_filename) as f:
        df = pd.read_csv(f, encoding='latin1')

print("อ่านข้อมูลสำเร็จ! ขนาดข้อมูล:", df.shape)

# 3. สร้างกราฟแสดงผล
plt.figure(figsize=(8, 5))
plt.scatter(df['Likes'], df['Impressions'], color='purple', alpha=0.6)
plt.title('Likes vs Impressions')
plt.xlabel('Likes')
plt.ylabel('Impressions')
plt.grid(True)
plt.show()

In [ ]:
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# 1. โหลดข้อมูล
zip_path = r'C:\Users\User\Downloads\archive.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    csv_filename = [f for f in z.namelist() if f.endswith('.csv')][0]
    with z.open(csv_filename) as f:
        df = pd.read_csv(f, encoding='latin1')

features = ['Likes', 'Saves', 'Comments', 'Shares', 'Profile Visits', 'Follows']
X = df[features]
y = df['Impressions']

# 2. เทรนโมเดล Linear Regression
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

# 3. พล็อตกราฟ 4 มุมมอง
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# กราฟที่ 1: อิทธิพลของแต่ละ Metric ต่อ Impressions (Coefficients)
coefs = pd.Series(model.coef_, index=features).sort_values()
axes[0, 0].barh(coefs.index, coefs.values, color='#4C72B0')
axes[0, 0].set_title('1. Impact of Metrics on Impressions (Coefficients)', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Coefficient Value')

# กราฟที่ 2: เปรียบเทียบยอดจริง VS ยอดที่โมเดลทำนายได้
axes[0, 1].scatter(y, y_pred, alpha=0.6, color='#55A868')
axes[0, 1].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
axes[0, 1].set_title('2. Actual vs Predicted Impressions', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Actual Impressions')
axes[0, 1].set_ylabel('Predicted Impressions')

# กราฟที่ 3: Correlation Matrix (ความสัมพันธ์ระหว่างตัวแปร)
corr = df[features + ['Impressions']].corr()
im = axes[1, 0].imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
axes[1, 0].set_title('3. Correlation Matrix', fontsize=11, fontweight='bold')
axes[1, 0].set_xticks(range(len(corr.columns)))
axes[1, 0].set_yticks(range(len(corr.columns)))
axes[1, 0].set_xticklabels(corr.columns, rotation=45, ha='right')
axes[1, 0].set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        axes[1, 0].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", 
                        color="black" if abs(corr.iloc[i, j]) < 0.7 else "white", fontsize=8)
fig.colorbar(im, ax=axes[1, 0])

# กราฟที่ 4: ค่าเฉลี่ยการมีส่วนร่วมต่อโพสต์
engagement_means = df[['Likes', 'Saves', 'Profile Visits', 'Follows']].mean()
axes[1, 1].bar(engagement_means.index, engagement_means.values, color=['#C44E52', '#8172B0', '#CCB974', '#64B5CD'])
axes[1, 1].set_title('4. Average Engagement per Post', fontsize=11, fontweight='bold')
axes[1, 1].set_ylabel('Average Count')

plt.tight_layout()
plt.show()